Goal for today: Create an SP400 addition to the report

In [5]:
 # Setup paths and imports

import sys, os
from datetime import datetime
import pandas as pd
# Define key directories

NOTEBOOK_DIR = os.getcwd()

BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
SRC_DIR = os.path.join(BASE_DIR, "src")
REPORT_DIR = os.path.join(BASE_DIR, "reports")

sys.path.insert(0, SRC_DIR)
os.makedirs(REPORT_DIR, exist_ok=True)

# Import screener modules
from prices import get_target_dates, download_all_required_price_data
from ranking import get_price_snapshots, compute_returns_and_ranks, store_top10_picks, store_top10_mdy_picks
from report import cache_company_data
from emailer import format_html_email, format_dual_html_email
import allocations

# Function to generate a single HTML report\n",
def generate_html_for_date(friday_str):
    print(f"⏳ Generating report for {friday_str}")

    anchor = pd.Timestamp(friday_str)      
    download_all_required_price_data(today = anchor)
    target_dates = get_target_dates(today=anchor)
    df, resolved = get_price_snapshots(target_dates)
    ranks = compute_returns_and_ranks(df, resolved)
    top10 = store_top10_picks(ranks, run_date=anchor)
    
    if top10.empty:
        print("⚠️ No top 10 results to include.")
        return

    tickers = top10["ticker"].tolist()
    cache_company_data(tickers)
    html_content = format_html_email(top10, report_date=anchor)
    
    output_path = os.path.join(REPORT_DIR, f"momentum_{friday_str}.html")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print(f"✅ Report written: {output_path}")

    #example usage
    #generate_html_for_date("2025-07-11")

In [ ]:
def generate_core_html_for_date(friday_str):
    print(f"⏳ Generating core report for {friday_str}")
    anchor = pd.Timestamp(friday_str)
    download_all_required_price_data(today = anchor)
        # Get target dates for the given anchor date
        # fetch and store group prices
        # fetch and store index prices

    
    target_dates = get_target_dates(today=anchor)
    spydf, resolved = get_price_snapshots(target_dates)
    mdydf, resolved = get_price_snapshots(target_dates, index_type="sp400")

    spyranks = compute_returns_and_ranks(spydf, resolved)
    mdyranks = compute_returns_and_ranks(mdydf, resolved)

    top10spy = store_top10_picks(spyranks, run_date=anchor)
    if top10spy.empty:
        print("⚠️ No top 10 results from SPY to include.")
        return

    top10mdy = store_top10_mdy_picks(mdyranks, run_date=anchor)
    if top10mdy.empty:
        print("⚠️ No top 10 results from MDY to include.")
        return

    spytickers = top10spy["ticker"].tolist()
    mdytickers = top10mdy["ticker"].tolist()
    
    cache_company_data(spytickers)
    cache_company_data(mdytickers)

    html_content = format_dual_html_email(top10spy, top10mdy, report_date=anchor)

    
    output_path = os.path.join(REPORT_DIR, f"momentum_dual_{friday_str}.html")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print(f"✅ Report written: {output_path}")

    #example usage
    #generate_html_for_date("2025-07-11")

generate_core_html_for_date("2025-06-27")
generate_core_html_for_date("2025-07-04")
generate_core_html_for_date("2025-07-11")


⏳ Generating core report for 2025-06-27
Skipping 2025-06-26 — already in DB
Skipping SPX for 2025-06-26 — already in DB
Fetching grouped prices for 2025-06-19...
No data for 2025-06-19 — trying previous weekday...
Skipping 2025-06-18 — already in DB
Fetching SPX price for 2025-06-19...
No SPX data for 2025-06-19 — trying previous weekday...
Skipping SPX for 2025-06-18 — already in DB
Skipping 2024-06-26 — already in DB
Skipping SPX for 2024-06-26 — already in DB
Fetching grouped prices for 2025-05-26...
No data for 2025-05-26 — trying previous weekday...
Skipping 2025-05-23 — already in DB
Fetching SPX price for 2025-05-26...
No SPX data for 2025-05-26 — trying previous weekday...
Skipping SPX for 2025-05-23 — already in DB
Fetching grouped prices for 2024-05-26...
No data for 2024-05-26 — trying previous weekday...
Skipping 2024-05-24 — already in DB
Fetching SPX price for 2024-05-26...
No SPX data for 2024-05-26 — trying previous weekday...
Skipping SPX for 2024-05-24 — already in DB

NameError: name 'spy_top10_df' is not defined

In [ ]:
from emailer import format_dual_html_email, send_email_via_sendgrid

# display(html)   # Jupyter preview

send_email_via_sendgrid(
    subject="Weekly Momentum Report – 15 Jul 2025",
    html_body=html,
    from_email=FROM_EMAIL,
    to_email=TO_EMAIL,
)


In [7]:
DB_PATH = '../data/market_data.sqlite'


with sqlite3.connect(DB_PATH) as conn:
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()

# Show the table names
for table in tables:
    print(table)

('index_allocations',)
('daily_prices',)
('index_constituents',)
('company_metadata',)
('company_news',)
('top10_picks',)


In [17]:
import sqlite3

with sqlite3.connect(DB_PATH) as conn:
    # Grab distinct values of index_type
    index_types = [
        row[0] for row in conn.execute(
            "SELECT DISTINCT index_type FROM index_constituents;"
        ).fetchall()
    ]

print(index_types)   # ['SP500', 'SP400', ...]


['sp500']


In [14]:
import importlib
import sys
from pathlib import Path

# -- (optional) add project root to sys.path so Python can find the script --
project_root = Path.cwd()  # adjust if your notebook sits elsewhere
sys.path.append(str(project_root))

import allocations   # now Python loads allocations.py

# Rerun the module if you edit it while the notebook is open:
importlib.reload(allocations)

# Option 1 – run the high‑level wrapper exactly once
allocations.update_index_allocations()

# Option 2 – do one ETF at a time
# allocations.download_spdr_holdings(allocations.SPY_URL, "spy_holdings.xlsx")
# allocations.parse_and_store_allocation("spy_holdings.xlsx", "sp500")


Saved file: spy_holdings.xlsx
Attempting to connect to database at: /Users/zacseidel/Documents/GitHub/momentum-screener/data/market_data.sqlite
Columns in spy_holdings.xlsx: ['Name', 'Ticker', 'Identifier', 'SEDOL', 'Weight', 'Sector', 'Shares Held', 'Local Currency']
Stored 504 rows for sp500
Saved file: mdy_holdings.xlsx
Attempting to connect to database at: /Users/zacseidel/Documents/GitHub/momentum-screener/data/market_data.sqlite
Columns in mdy_holdings.xlsx: ['Name', 'Ticker', 'Identifier', 'SEDOL', 'Weight', 'Sector', 'Shares Held', 'Local Currency']
Stored 402 rows for sp400


In [ ]:
import os
import sqlite3
import pandas as pd
from pathlib import Path
import allocations    # assuming you already imported or executed the script

# 1️⃣ confirm we’re looking at the database you expect
print("DB_PATH:", allocations.DB_PATH)
print("Exists on disk?", Path(allocations.DB_PATH).is_file())

with sqlite3.connect(allocations.DB_PATH) as conn:
    # 2️⃣ check which tables are present
    tables = conn.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type='table';
    """).fetchall()
    print("Tables:", tables)

    # 3️⃣ confirm the row count
    total_rows = conn.execute("SELECT COUNT(*) FROM index_allocations;").fetchone()[0]
    print(f"index_allocations has {total_rows:,} rows")

    
    # 4️⃣ look at counts by date & index_type (should match the 504 / 402 prints)
    summary = pd.read_sql("""
        SELECT date, index_type, COUNT(*) AS n
        FROM index_allocations
        GROUP BY date, index_type
        ORDER BY date DESC, index_type;
    """, conn)
    display(summary.tail())   # show the most‑recent dates

    # 5️⃣ peek at the newest 10 rows
    recent = pd.read_sql("""
        SELECT *
        FROM index_allocations
        ORDER BY ROWID DESC
        LIMIT 10;
    """, conn)
    display(recent)


DB_PATH: /Users/zacseidel/Documents/GitHub/momentum-screener/data/market_data.sqlite
Exists on disk? True
Tables: [('index_allocations',), ('daily_prices',), ('index_constituents',), ('company_metadata',), ('company_news',), ('top10_picks',)]
index_allocations has 1,812 rows


,date,index_type,n
0,2025-07-16,sp400,402
1,2025-07-16,sp500,504
2,2025-05-09,sp400,402
3,2025-05-09,sp500,504


,ticker,company,index_type,date,weight
0,UA,Under Armour Inc. Class C,sp400,2025-07-16,0.027660
1,UAA,Under Armour Inc. Class A,sp400,2025-07-16,0.043585
2,SAM,Boston Beer Company Inc. Class A,sp400,2025-07-16,0.053243
3,WEN,Wendy's Company,sp400,2025-07-16,0.057293
4,GEF,Greif Inc Class A,sp400,2025-07-16,0.057409
5,SHC,Sotera Health Company,sp400,2025-07-16,0.058951
6,PPC,Pilgrim's Pride Corporation,sp400,2025-07-16,0.059984
7,SRPT,Sarepta Therapeutics Inc.,sp400,2025-07-16,0.060481
8,COLM,Columbia Sportswear Company,sp400,2025-07-16,0.060735
9,COTY,Coty Inc. Class A,sp400,2025-07-16,0.060916


In [7]:
import sqlite3, pandas as pd
DB_PATH = '../data/market_data.sqlite'

print("DB_PATH:", allocations.DB_PATH)


def inspect_index(index_type="sp400", db_path=DB_PATH, sample=5):
    with sqlite3.connect(db_path) as conn:
        # A) what values actually exist?
        actual_types = pd.read_sql("SELECT DISTINCT index_type FROM index_allocations", conn)
        print("index_allocations.index_type values:\n", actual_types["index_type"].tolist())

        # B) how many tickers match the string you’re passing in?
        tickers = pd.read_sql(
            "SELECT distinct ticker FROM index_allocations WHERE index_type = ?",
            conn, params=[index_type]
        )
        print(f"{len(tickers)} tickers tagged '{index_type}'")

        # C) do those tickers exist in daily_prices at all?
        present = pd.read_sql(
            f"""
            SELECT COUNT(DISTINCT ticker) AS in_prices
            FROM daily_prices
            WHERE ticker IN ({','.join(['?']*len(tickers))})
            """,
            conn, params=tickers["ticker"].tolist()
        ).iloc[0, 0]
        print(f"{present} of them have price rows in daily_prices")

        # D) peek at a handful
        print("Sample tickers:", tickers["ticker"].head(sample).tolist())

inspect_index("sp400")


DB_PATH: /Users/zacseidel/Documents/GitHub/momentum-screener/data/market_data.sqlite
index_allocations.index_type values:
 ['sp500', 'sp400']
406 tickers tagged 'sp400'
405 of them have price rows in daily_prices
Sample tickers: ['AA', 'AAL', 'AAON', 'ACHC', 'ACI']


In [3]:
import sqlite3
from pathlib import Path

DB_PATH = Path("../data/market_data.sqlite").resolve()  # adjust if needed

create_sql = """
CREATE TABLE IF NOT EXISTS top10_mdy (
    ticker TEXT NOT NULL,
    date   DATE NOT NULL,
    current_return     TEXT,
    last_month_return  TEXT,
    last_week_return   TEXT,
    current_rank       REAL,
    last_month_rank    REAL,
    rank_change        REAL,
    PRIMARY KEY (ticker, date)
);
"""

with sqlite3.connect(DB_PATH) as conn:
    conn.execute(create_sql)
    conn.commit()

print("✅ `top10_mdy` table now present in", DB_PATH)


✅ `top10_mdy` table now present in /Users/zacseidel/Documents/GitHub/momentum-screener/data/market_data.sqlite
